## Part 3: Feature Engineering

The goal of this notebook is to generate features for use in training predictive models using variables contained in the dataset. 

This will involve: 
- Regularizing features so that they are all roughly the same magnitude.
- Calculating the percent change of any features that have a strong trend over time. For these types of features, the magnitude of the change from month to month can have very different importance depending on its current value. For example, QQQ increasing from $1 to $2 is much more interesting than it increasing from $100 to $101.
- Defining the target variables for various predictors I want to train. 
    - In the case of defining market health, I will consider two target variables:
        - the GDP next quarter
        - the GDP in a year
    - I will also train a model to predict the value of QQQ in a month's time and see if I can use that to create an QQQ buy strategy that outcompetes a simple buy-and-hold strategy. That work will be done in the 4th notebook.

--------

Each feature gets one of three treatments before being z-scored, implemented in `macro_etf/features.py::build_features()`:
- Features with a persistent secular trend (cpi, retail sales, etc) are converted to percent change, since the rate of change is usually more informative than the raw level.
- Features that are non-negative and heavily right-tailed with rare extreme spikes (initial claims, VIX, trading volumes, personal savings rate) get a log1p transform first, so a single outlier month (mostly COVID-driven) doesn't dominate a linear model's fit.
- Everything else hovers around a fairly stable range with no strong trend, so it's left as a raw level and z-scored directly.

---

In [21]:
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from macro_etf.common import PROCESSED_DATA_DIR, MODELLING_TARGETS
from macro_etf.features import build_features

data_fn = "processed_market_data.csv"


### 3.1: Load data

In [22]:
df = pd.read_csv(PROCESSED_DATA_DIR / data_fn, index_col=0)
df.index = pd.to_datetime(df.index)
df.head()

,yield_spread,credit_spread,financial_stress,initial_claims,cpi,unemployment,fed_funds_rate,industrial_production,retail_sales,consumer_sentiment,...,job_openings,credit_card_delinquency,close_spy,close_qqq,close_vix,close_oil,volume_spy,volume_qqq,volume_oil,real_gdp
2001-02-28,0.51,2.88,0.437625,371250.0,175.6,4.2,5.49,91.9020,247339.0,94.7,...,5088.0,4.81,78.485298,39.983036,28.350000,27.420000,14825800.0,102187900.0,82364.0,14229.765
2001-03-31,0.75,3.04,0.675540,387200.0,176.0,4.2,5.31,91.3034,247289.0,90.6,...,5234.0,4.81,74.087158,32.989159,28.639999,26.400000,9183600.0,63661300.0,63947.0,14183.120
2001-04-30,1.05,2.73,0.800700,396750.0,176.1,4.3,4.80,91.1162,244514.0,91.5,...,5097.0,4.81,80.417152,38.887619,25.480000,28.459999,10766900.0,73859100.0,40223.0,14183.120
2001-05-31,1.21,2.65,0.387725,394500.0,176.4,4.4,4.21,90.7891,249113.0,88.4,...,4762.0,4.94,79.966370,37.691078,22.639999,28.370001,9874200.0,68018300.0,101694.0,14183.120
2001-06-30,1.17,2.65,0.420820,397200.0,177.3,4.3,3.97,90.3555,250250.0,92.0,...,4615.0,4.94,78.060844,38.508430,19.059999,26.250000,9824200.0,59719600.0,86303.0,14271.694


### 3.2: Define a feature matrix and prediction targets

In [23]:
# Let's start by defining the feature matrix as the entire unaltered feature set
X = df.copy()

for target, target_param in MODELLING_TARGETS.items():
    X[f'target_{target}'] = (
        X[target_param['variable']]
            .pct_change(periods=target_param['date_gap']) #Take the percent change as that is more informative for variables with a trend.
            .shift(periods=-target_param['date_gap']) #shift the variables so that they correspond to the date gap specified
    )

The features should be normalized / regularized / modified before being passed as training variables to a model. Features that trend upwards over time (e.g., cpi, retail sales, etc) can be handled poorly depending on choice of model, and what's often more important for these is the percent change from month to month rather than the raw number, e.g., an ETF increasing in value from 1 to 2 is a lot more interesting than an increase of 100 to 101.

Some features are non-negative and heavily right-tailed with rare extreme spikes (e.g., initial claims, VIX, trading volumes, personal savings rate), primarily from the COVID pandemic. Left as raw levels, those outlier months would dominate a linear model's fit once z-scored, so these get a log1p transform first to compress their influence while preserving relative ordering. By preserving the ordering, tree-models should be relatively unaffected by this scaling.

Both of these transforms are implemented in `build_features()`.

In [24]:
X = build_features(X)

In [25]:
# Due to calculating percent changes and shifting, 
# there will now be some nan entries at the end of
# the dataset. Let's remove them.
X = X.dropna()

In [26]:
# Display the results
display(X.head())
display(X.tail())

,yield_spread,credit_spread,financial_stress,initial_claims,unemployment,fed_funds_rate,industrial_production,consumer_sentiment,personal_savings_rate,housing_starts,...,target_gdp_12mo,target_qqq_1mo,target_qqq_3mo,cpi_percent_change,retail_sales_percent_change,m2_percent_change,close_spy_percent_change,close_qqq_percent_change,close_oil_percent_change,real_gdp_percent_change
2001-05-31,1.21,2.65,0.387725,12.885377,4.4,4.21,90.7891,88.4,1.757858,1649.0,...,0.013373,0.021686,-0.181087,0.001704,0.018809,0.012670,-0.005606,-0.030769,-0.003162,-0.003278
2001-06-30,1.17,2.65,0.420820,12.892198,4.3,3.97,90.3555,92.0,1.667707,1605.0,...,0.013254,-0.086214,-0.365865,0.005102,0.004564,-0.000370,-0.023829,0.021686,-0.074727,0.006245
2001-07-31,1.28,2.78,0.408925,12.894210,4.5,3.77,89.8784,92.6,1.648659,1636.0,...,0.013254,-0.122845,-0.188219,0.002256,-0.005650,0.008351,-0.010196,-0.086214,0.003810,0.006245
2001-08-31,1.21,2.90,0.360120,12.894210,4.6,3.65,89.3028,92.4,1.856298,1670.0,...,0.013254,-0.208845,0.082446,-0.001688,-0.006028,0.005772,-0.059332,-0.122845,0.032258,0.006245
2001-09-30,1.74,3.42,1.194475,12.983104,4.9,3.07,89.2003,91.5,2.014903,1567.0,...,0.021465,0.169773,0.342650,0.000000,0.009113,0.006488,-0.081630,-0.208845,-0.138603,-0.004006


,yield_spread,credit_spread,financial_stress,initial_claims,unemployment,fed_funds_rate,industrial_production,consumer_sentiment,personal_savings_rate,housing_starts,...,target_gdp_12mo,target_qqq_1mo,target_qqq_3mo,cpi_percent_change,retail_sales_percent_change,m2_percent_change,close_spy_percent_change,close_qqq_percent_change,close_oil_percent_change,real_gdp_percent_change
2025-02-28,0.25,1.57,-0.701125,12.323860,4.0,4.33,100.0647,71.7,1.808289,1353.0,...,0.019893,-0.075862,0.023052,0.004273,-0.010564,0.002900,-0.012695,-0.027035,-0.038191,0.004599
2025-03-31,0.34,1.76,-0.453250,12.314932,4.2,4.33,101.0993,64.7,1.824549,1491.0,...,0.026847,0.013968,0.177727,0.002251,0.002424,0.003030,-0.055719,-0.075862,0.024656,-0.001625
2025-04-30,0.57,1.96,0.060925,12.327188,4.2,4.33,101.0404,57.0,1.808289,1346.0,...,0.026847,0.091783,0.189654,0.000332,0.014245,0.003711,-0.008670,0.013968,-0.185646,-0.001625
2025-05-31,0.52,1.84,-0.543680,12.353635,4.2,4.33,101.1279,52.2,1.871802,1400.0,...,0.026847,0.063858,0.100038,0.001617,-0.001836,0.003789,0.062845,0.091783,0.044322,-0.001625
2025-06-30,0.52,1.75,-0.731225,12.384223,4.3,4.33,100.9655,52.2,1.774952,1289.0,...,0.017224,0.024237,0.089598,0.000993,-0.012386,0.002668,0.051386,0.063858,0.071064,0.009460


### 3.3 Split the data into training and test sets

In [27]:
# Let's split the data into training and test data
train_prop = 0.7
test_prop = 1.0 - train_prop
n_train = int(train_prop*X.shape[0])
n_test = X.shape[0] - n_train
train_inds = np.arange(0,n_train)
test_inds = np.arange(n_train, X.shape[0])


split_labels = ['train']*n_train + ['test']*n_test
X['split'] = split_labels

### 3.4: Regularize the features

In [ ]:
from sklearn.preprocessing import StandardScaler

# Train a scaler using the training dataset and then apply the scaler to both the training and test datasets
#   We do this split so that information contained in the training set isn't accidentally incorporated into the test set.
#   This can happen when calculating a feature's z-score if all values are used to calculate the mean and std.
#   I also need to be careful to not scale the target variables, as otherwise I will have to de-scale them and that would be annoying

target_cols = [col for col in X.columns if 'target' in col] + ['split']
non_target_cols = list (set(X.columns) - set(target_cols))

X_train = X.query('split == "train"')[non_target_cols]
X_test = X.query('split == "test"')[non_target_cols]
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = pd.DataFrame(
    scaler.transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

X_scaled = pd.concat(
    [
        X_train_scaled, 
        X_test_scaled
    ], 
    axis=0
)
X_scaled = pd.concat(
    [X_scaled, X[target_cols]], axis=1
)

### 3.5: Save training / test datasets + target variables

In [30]:
# I'm going to try to keep things pandasy and output one big 
# dataframe containing the required info.

X_scaled.to_csv(PROCESSED_DATA_DIR / 'feature_matrix.csv')